In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# 定义要进行PCA的植被指数列
vi_columns = [
    'CMRI_mean_mangrove',
    'EMVI_mean_mangrove',
    'EVI_mean_mangrove',
    'MVI_mean_mangrove',
    'NDVI_mean_mangrove',
    'kNDVI_mean_mangrove'
]

# 读取数据
try:
    df = pd.read_csv('final_environment_interpolated.csv')
    print("文件 'final_environment_interpolated.csv' 读取成功。")
    
    # 新增功能：删除所有年份中 vi_columns 全部为空值的网格
    if 'GridID' in df.columns:
        print("\n正在检查并清理无效网格...")
        initial_grid_count = df['GridID'].nunique()
        
        # 对于每个GridID，如果所有年份的所选植被指数均为NaN，则标记为待删除
        # count()会计算非NaN的数量，如果一个GridID所有行在这些列上的非NaN总数为0，即为全空
        valid_obs_per_grid = df.groupby('GridID')[vi_columns].count().sum(axis=1)
        invalid_grids = valid_obs_per_grid[valid_obs_per_grid == 0].index
        
        if len(invalid_grids) > 0:
            df = df[~df['GridID'].isin(invalid_grids)].copy()
            df.reset_index(drop=True, inplace=True)
            print(f"删除了 {len(invalid_grids)} 个在所有年份均无植被指数数据的网格。")
            print(f"剩余网格数：{df['GridID'].nunique()} (原始网格数：{initial_grid_count})")
        else:
            print("所有网格均包含有效的植被指数数据，无需删除。")
    else:
        print("\n警告：未检测到 'GridID' 列，跳过网格清理步骤。")

    # 提取植被指数数据
    vi_data = df.loc[:, vi_columns].copy()
    
    # 检查缺失值
    missing_count = vi_data.isnull().sum().sum()
    if missing_count > 0:
        print(f"\n警告：检测到 {missing_count} 个缺失值。")
        print("缺失值详情：")
        print(vi_data.isnull().sum())
        
        # 删除含有缺失值的行
        valid_indices = vi_data.dropna().index
        vi_data_clean = vi_data.loc[valid_indices].copy()
        print(f"\n跳过 {len(df) - len(valid_indices)} 行含缺失值的数据。")
        print(f"有效数据行数：{len(valid_indices)}")
    else:
        print("\n未检测到缺失值。")
        valid_indices = vi_data.index
        vi_data_clean = vi_data
    
    # 数据标准化
    scaler = StandardScaler()
    vi_data_scaled = scaler.fit_transform(vi_data_clean)
    print("植被指数数据已标准化。")

    # 执行PCA
    pca = PCA(n_components=1)
    principal_component = pca.fit_transform(vi_data_scaled)
    
    # 获取PCA载荷
    loadings = pca.components_[0]
    print("\nPCA第一主成分载荷：")
    for col, loading in zip(vi_columns, loadings):
        print(f"  {col}: {loading:.4f}")
    
    # 方向校正
    avg_loading = np.mean(loadings)
    if avg_loading < 0:
        print("\n检测到主成分方向与植被健康度负相关，正在翻转方向...")
        principal_component = -principal_component
        loadings = -loadings
        print("方向已校正，MHI现在与植被健康度正相关。")
    else:
        print("\n主成分方向正常，MHI与植被健康度正相关。")
    
    print("\n校正后的载荷：")
    for col, loading in zip(vi_columns, loadings):
        print(f"  {col}: {loading:.4f}")
    
    print("主成分分析（PCA）执行完毕。")

    # 创建MHI列
    df['MHI'] = np.nan
    df.loc[valid_indices, 'MHI'] = principal_component.flatten()
    print("已创建 'MHI' 列。")

    # === 红树林动态分类 ===
    print("\n开始进行红树林动态分类...")
    
    if 'GridID' not in df.columns or 'year' not in df.columns:
        print("警告：未找到'GridID'或'year'列，无法进行动态分类。")
        df['mangrove_class'] = 'unknown'
    else:
        # 初始化分类列
        df['mangrove_class'] = 'unknown'
        
        # 获取所有年份范围
        all_years = sorted(df['year'].unique())
        total_years = len(all_years)
        early_years = [2000]
        late_years = [2024]
        
        print(f"时间范围：{all_years[0]} - {all_years[-1]}")
        print(f"早期年份：{early_years}")
        print(f"后期年份：{late_years}")
        
        # 按GridID分组分析
        for grid_id, group in df.groupby('GridID'):
            indices = group.index
            
            # 检查早期和后期是否有MHI值
            early_data = group[group['year'].isin(early_years)]
            late_data = group[group['year'].isin(late_years)]
            
            early_has_mhi = early_data['MHI'].notna().any()
            late_has_mhi = late_data['MHI'].notna().any()
            all_has_mhi = group['MHI'].notna().all()
            
            # 分类逻辑
            if all_has_mhi:
                df.loc[indices, 'mangrove_class'] = '1'#持续红树林
            elif not early_has_mhi and late_has_mhi:
                df.loc[indices, 'mangrove_class'] = '2'#新生红树林
            elif early_has_mhi and not late_has_mhi:
                df.loc[indices, 'mangrove_class'] = '0'#消失红树林
            else:
                df.loc[indices, 'mangrove_class'] = '3'#不稳定红树林
        
        # 统计各类别
        print("\n红树林动态分类结果：")
        class_counts = df.groupby('mangrove_class')['GridID'].nunique()
        total_grids = df['GridID'].nunique()
        for class_name, count in class_counts.items():
            percentage = (count / total_grids) * 100
            print(f"  {class_name}: {count} 个网格 ({percentage:.2f}%)")
        
        print(f"\n总网格数：{total_grids}")

    # 保存结果
    output_filename = 'final_environment_with_MHI.csv'
    df.to_csv(output_filename, index=False)
    print(f"\n处理完成！更新后的数据已保存到 '{output_filename}'。")
    print("\n新数据框的前几行（包含MHI和分类）：")
    print(df[['GridID', 'year', 'MHI', 'mangrove_class']].head(10))
    print(f"\nMHI 统计信息：")
    print(f"- 有效值数量：{df['MHI'].notna().sum()}")
    print(f"- 缺失值数量：{df['MHI'].isna().sum()}")
    if df['MHI'].notna().any():
        print(f"- MHI均值：{df['MHI'].mean():.4f}")
        print(f"- MHI标准差：{df['MHI'].std():.4f}")
        print(f"- MHI范围：[{df['MHI'].min():.4f}, {df['MHI'].max():.4f}]")

except FileNotFoundError:
    print("错误：'final_environment_interpolated.csv' 文件未找到。请确保文件位于正确的目录中。")
except Exception as e:
    print(f"处理过程中发生错误：{e}")

In [ ]:
pc1_explained_variance = pca.explained_variance_[0]
pc1_explained_variance_ratio = pca.explained_variance_ratio_[0]

print(f"PC1 解释方差: {pc1_explained_variance:.4f}")
print(f"PC1 解释方差占比: {pc1_explained_variance_ratio:.4%}")